# Лабораторная работа № 1
## Выполнение разведочного анализа больших данных с использованием Apache Spark

### Тема работы
Разведочный анализ датасета Lending Club Loans Data с использованием Apache Spark, HDFS и Apache Iceberg.

### Цель работы
Освоить основные этапы работы с большими данными:
- чтение данных из HDFS;
- предварительная подготовка данных;
- приведение столбцов к корректным типам;
- очистка данных;
- выполнение базового разведочного анализа;
- сохранение подготовленного датасета в таблицу Apache Iceberg.


Подключаем необходимые библиотеки.


In [1]:
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.sql.functions import (
    col,
    regexp_replace,
    regexp_extract,
    to_date,
    year,
    month,
    when,
    count,
    avg,
    round
)


Сформируем объект конфигурации для Apache Spark, указав необходимые параметры.


In [2]:
def create_spark_configuration() -> SparkConf:
    """
    Создаёт и конфигурирует экземпляр SparkConf для приложения Spark.
    Адаптация для локального выполнения в docker (local[*]) с доступом к HDFS.
    """
    user_name = "jovyan"

    conf = SparkConf()
    conf.setAppName("Lab1_Lending_Club_EDA")
    conf.setMaster("local[*]")

    conf.set("spark.sql.adaptive.enabled", "true")

    conf.set("spark.executor.memory", "6g")
    conf.set("spark.executor.cores", "2")
    conf.set("spark.executor.instances", "1")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "1")

    conf.set("spark.hadoop.fs.defaultFS", "hdfs://hadoop-namenode:9820")
    conf.set("spark.hadoop.dfs.client.use.datanode.hostname", "true")

    conf.set("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")
    conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///user/{user_name}/warehouse")
    conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")

    return conf


Создаём объект конфигурации и SparkSession.


In [3]:
conf = create_spark_configuration()
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark


В HDFS файл уже загружен по адресу:
`hdfs://hadoop-namenode:9820/data/lending_club/accepted_2007_to_2017.csv`


In [4]:
path = "hdfs://hadoop-namenode:9820/data/lending_club/accepted_2007_to_2017.csv"

df = (
    spark.read.format("csv")
    .option("header", "true")
    .load(path)
)


Выводим фрагмент датафрейма на экран.


In [5]:
df.limit(10).toPandas().style


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,38098114,None,15000.0,15000.0,15000.0,60 months,12.39,336.64,C,C1,MANAGEMENT,10+ years,RENT,78000.0,Source Verified,Dec-2014,Fully Paid,n,None,debt_consolidation,Debt consolidation,235xx,VA,12.03,0.0,Aug-1994,750.0,754.0,0.0,None,None,6.0,0.0,138008.0,29.0,17.0,w,0.0,0.0,17392.37,17392.37,15000.0,2392.37,0.0,0.0,0.0,Jun-2016,12017.81,None,Nov-2017,684.0,680.0,0.0,None,1.0,Individual,None,None,None,0.0,0.0,149140.0,None,None,None,None,None,None,None,None,None,None,None,184500.0,None,None,None,5.0,29828.0,9525.0,4.7,0.0,0.0,103.0,244.0,1.0,1.0,0.0,47.0,None,None,None,0.0,1.0,4.0,1.0,2.0,8.0,5.0,9.0,4.0,6.0,0.0,0.0,0.0,4.0,100.0,0.0,0.0,0.0,196500.0,149140.0,10000.0,12000.0,None,None,None,None,None,None,None,None,None,None,None,None,None,N,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Cash,N,None,None,None,None,None,None
1,36805548,None,10400.0,10400.0,10400.0,36 months,6.99,321.08,A,A3,Truck Driver Delivery Personel,8 years,MORTGAGE,58000.0,Not Verified,Dec-2014,Charged Off,n,None,credit_card,Credit card refinancing,937xx,CA,14.92,0.0,Sep-1989,710.0,714.0,2.0,42.0,None,17.0,0.0,6133.0,31.6,36.0,w,0.0,0.0,6611.69,6611.69,5217.75,872.67,0.0,521.27,93.8286,Aug-2016,321.08,None,Feb-2017,564.0,560.0,0.0,59.0,1.0,Individual,None,None,None,0.0,0.0,162110.0,None,None,None,None,None,None,None,None,None,None,None,19400.0,None,None,None,7.0,9536.0,7599.0,41.5,0.0,0.0,76.0,290.0,1.0,1.0,1.0,5.0,42.0,1.0,42.0,4.0,6.0,9.0,7.0,18.0,2.0,14.0,32.0,9.0,17.0,0.0,0.0,0.0,4.0,83.3,14.3,0.0,0.0,179407.0,15030.0,13000.0,11325.0,None,None,None,None,None,N

Для анализа оставим только 10 исходных столбцов, чтобы ноутбук был компактнее и проще для объяснения.


In [6]:
df = df.select(
    "id",
    "loan_amnt",
    "term",
    "int_rate",
    "grade",
    "annual_inc",
    "issue_d",
    "loan_status",
    "purpose",
    "dti"
)

df.show()


+--------+---------+----------+--------+-----+----------+--------+-----------+------------------+-----+
|      id|loan_amnt|      term|int_rate|grade|annual_inc| issue_d|loan_status|           purpose|  dti|
+--------+---------+----------+--------+-----+----------+--------+-----------+------------------+-----+
|38098114|  15000.0| 60 months|   12.39|    C|   78000.0|Dec-2014| Fully Paid|debt_consolidation|12.03|
|36805548|  10400.0| 36 months|    6.99|    A|   58000.0|Dec-2014|Charged Off|       credit_card|14.92|
|37842129|  21425.0| 60 months|   15.59|    D|   63800.0|Dec-2014| Fully Paid|       credit_card|18.49|
|37612354|  12800.0| 60 months|   17.14|    D|  125000.0|Dec-2014|    Current|               car| 8.31|
|37662224|   7650.0| 36 months|   13.66|    C|   50000.0|Dec-2014|Charged Off|debt_consolidation|34.81|
|37822187|   9600.0| 36 months|   13.66|    C|   69000.0|Dec-2014| Fully Paid|debt_consolidation|25.81|
|37741884|   2500.0| 36 months|   11.99|    B|   89000.0|Dec-201

Выведем на экран метаданные датасета.


In [7]:
df.printSchema()


root
 |-- id: string (nullable = true)
 |-- loan_amnt: string (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- dti: string (nullable = true)



Приведём столбцы к корректным типам и сформируем несколько дополнительных признаков:
- `term_months` — срок кредита в месяцах;
- `issue_date` — дата выдачи кредита;
- `issue_year`, `issue_month` — год и месяц выдачи;
- `is_default` — бинарный признак проблемного займа.


In [8]:
def transform_dataframe(data: DataFrame) -> DataFrame:
    """
    Приводит столбцы датафрейма к корректным типам и формирует дополнительные признаки.
    """
    data = data.withColumn("id", col("id").cast("int"))
    data = data.withColumn("loan_amnt", col("loan_amnt").cast("double"))
    data = data.withColumn("annual_inc", col("annual_inc").cast("double"))
    data = data.withColumn("dti", col("dti").cast("double"))

    data = data.withColumn("int_rate", regexp_replace(col("int_rate"), "%", "").cast("double"))
    data = data.withColumn("term_months", regexp_extract(col("term"), r"(\d+)", 1).cast("int"))
    data = data.withColumn("issue_date", to_date(col("issue_d"), "MMM-yyyy"))
    data = data.withColumn("issue_year", year(col("issue_date")))
    data = data.withColumn("issue_month", month(col("issue_date")))

    data = data.withColumn(
        "is_default",
        when(
            col("loan_status").isin(
                "Charged Off",
                "Default",
                "Late (31-120 days)",
                "Late (16-30 days)"
            ),
            1
        ).otherwise(0)
    )

    return data


In [9]:
df = transform_dataframe(df)
df.show()


+--------+---------+----------+--------+-----+----------+--------+-----------+------------------+-----+-----------+----------+----------+-----------+----------+
|      id|loan_amnt|      term|int_rate|grade|annual_inc| issue_d|loan_status|           purpose|  dti|term_months|issue_date|issue_year|issue_month|is_default|
+--------+---------+----------+--------+-----+----------+--------+-----------+------------------+-----+-----------+----------+----------+-----------+----------+
|38098114|  15000.0| 60 months|   12.39|    C|   78000.0|Dec-2014| Fully Paid|debt_consolidation|12.03|         60|2014-12-01|      2014|         12|         0|
|36805548|  10400.0| 36 months|    6.99|    A|   58000.0|Dec-2014|Charged Off|       credit_card|14.92|         36|2014-12-01|      2014|         12|         1|
|37842129|  21425.0| 60 months|   15.59|    D|   63800.0|Dec-2014| Fully Paid|       credit_card|18.49|         60|2014-12-01|      2014|         12|         0|
|37612354|  12800.0| 60 months|   

In [10]:
df.printSchema()


root
 |-- id: integer (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- annual_inc: double (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- dti: double (nullable = true)
 |-- term_months: integer (nullable = true)
 |-- issue_date: date (nullable = true)
 |-- issue_year: integer (nullable = true)
 |-- issue_month: integer (nullable = true)
 |-- is_default: integer (nullable = false)



Удалим дубликаты и отбросим заведомо некорректные записи.


In [11]:
def clean_dataframe(data: DataFrame) -> DataFrame:
    """
    Удаляет дубликаты и отбрасывает заведомо некорректные значения.
    """
    data = data.dropDuplicates(["id"])
    data = data.filter(col("id").isNotNull())
    data = data.filter(col("loan_amnt").isNotNull())
    data = data.filter(col("int_rate").isNotNull())
    data = data.filter(col("annual_inc").isNotNull())
    data = data.filter(col("dti").isNotNull())
    data = data.filter(col("issue_date").isNotNull())
    data = data.filter(col("loan_status").isNotNull())

    data = data.filter(col("loan_amnt") > 0)
    data = data.filter(col("annual_inc") > 0)
    data = data.filter((col("int_rate") >= 0) & (col("int_rate") <= 40))
    data = data.filter((col("dti") >= 0) & (col("dti") <= 100))
    data = data.drop("term_months")

    return data


In [12]:
rows_before_cleaning = df.count()
df = clean_dataframe(df)
rows_after_cleaning = df.count()

print("Количество строк до очистки:", rows_before_cleaning)
print("Количество строк после очистки:", rows_after_cleaning)


Количество строк до очистки: 1646801
Количество строк после очистки: 1645516


In [13]:
database_name = "khramtsov_database"
table_name = "sobd_lab1_lending_club_table"


In [14]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS spark_catalog.{database_name}")


DataFrame[]

In [15]:
spark.sql(f"DROP TABLE IF EXISTS spark_catalog.{database_name}.{table_name}")

df.writeTo(
    f"spark_catalog.{database_name}.{table_name}"
).using("iceberg").create()


In [16]:
for table in spark.catalog.listTables(database_name):
    print(table.name)


sobd_lab1_lending_club_table
sobd_lab1_lending_club_processed


In [17]:
df_table = spark.table(f"{database_name}.{table_name}")
df_table.show(10)


+------+---------+----------+--------+-----+----------+--------+--------------------+------------------+-----+----------+----------+-----------+----------+
|    id|loan_amnt|      term|int_rate|grade|annual_inc| issue_d|         loan_status|           purpose|  dti|issue_date|issue_year|issue_month|is_default|
+------+---------+----------+--------+-----+----------+--------+--------------------+------------------+-----+----------+----------+-----------+----------+
| 54734|  25000.0| 36 months|   11.89|    B|   85000.0|Aug-2009|          Fully Paid|debt_consolidation|19.48|2009-08-01|      2009|          8|         0|
| 61390|   4000.0| 36 months|    7.88|    A|  148000.0|Feb-2010|          Fully Paid|       credit_card|16.98|2010-02-01|      2010|          2|         0|
| 65419|  16000.0| 36 months|    6.89|    A|   72000.0|Jun-2015|             Current|  home_improvement|23.43|2015-06-01|      2015|          6|         0|
| 70686|   5000.0| 36 months|    7.75|    A|   70000.0|Jun-2007|

In [18]:
spark.stop()
